<a href="https://colab.research.google.com/github/crd10lop/CodingLinealOptimizacion/blob/main/Programaci%C3%B3n_lineal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

def init_tablero(c, A, b):

    num_restricciones = len(b)
    num_variables = len(c)

    filas = num_restricciones + 1
    columnas = num_variables + num_restricciones + 1
    tablero = np.zeros((filas, columnas))

    tablero[:num_restricciones, :num_variables] = A
    tablero[:num_restricciones, num_variables:num_variables + num_restricciones] = np.eye(num_restricciones)
    tablero[:num_restricciones, -1] = b
    tablero[-1, :num_variables] = -c

    return tablero

def simplex(tablero, num_variables, num_restricciones):
    iteracion = 0

    while True:
        if np.all(tablero[-1, :-1] >= 0):
            print(f"\nSolucion Optima Encontrada en {iteracion} iteraciones")
            break

        columna_pivote = np.argmin(tablero[-1, :-1])

        rhs = tablero[:-1, -1]
        columna_entrada = tablero[:-1, columna_pivote]

        cocientes = np.divide(rhs, columna_entrada, out=np.full_like(rhs, np.inf), where=columna_entrada > 0)


        if np.all(np.isinf(cocientes)):
            raise ValueError("El problema no está acotado. No tiene solución finita.")

        fila_pivote = np.argmin(cocientes)
        elemento_pivote = tablero[fila_pivote, columna_pivote]
        tablero[fila_pivote, :] = tablero[fila_pivote, :] / elemento_pivote

        for f in range(tablero.shape[0]):
            if f != fila_pivote:
                factor = tablero[f, columna_pivote]
                tablero[f, :] = tablero[f, :] - factor * tablero[fila_pivote, :]

        iteracion += 1

    return tablero

def mostrar_resultados(tablero, num_variables, num_restricciones):
    cols_vars = [f"X{i+1}" for i in range(num_variables)]
    cols_holgura = [f"S{i+1}" for i in range(num_restricciones)]
    columnas = cols_vars + cols_holgura + ["RHS"]

    filas = [f"R{i+1}" for i in range(num_restricciones)] + ["Z"]

    df = pd.DataFrame(np.round(tablero, 4), columns=columnas, index=filas)
    print("\nTablero Final:")
    display(df)

    print("\nSolucion:")
    for i in range(num_variables):
        col = tablero[:, i]
        es_basica = np.sum(col == 1) == 1 and np.sum(col == 0) == len(col) - 1

        if es_basica:
            fila_indice = np.where(col == 1)[0][0]
            valor = tablero[fila_indice, -1]
        else:
            valor = 0.0
        print(f"X{i+1} = {valor:.4f}")

    print(f"Z Maximo = {tablero[-1, -1]:.4f}")




##DATOS ESTATICOS
"""

#función objetivo: Z
c = np.array([3.0, 5.0])
#restricciones: Matriz A
A = np.array([
    [1.0, 0.0],
    [0.0, 2.0],
    [3.0, 2.0]
])
#restricciones Vector b
b = np.array([4.0, 12.0, 18.0])
"""

##DATOS USUARIO
#c = np.array(int)

print("--- INGRESO DE DATOS ---")
# 1. Pedir coeficientes de Z
entrada_c = input("Ingrese los coeficientes de Z separados por espacio (Ej: 3 5): ")
c = np.array([float(x) for x in entrada_c.split()])

# 2. Pedir vector del lado derecho (b)
entrada_b = input("Ingrese los valores del lado derecho (RHS) separados por espacio (Ej: 4 12 18): ")
b = np.array([float(x) for x in entrada_b.split()])

# 3. Pedir la matriz de restricciones (A)
print("Ingrese los coeficientes de las restricciones fila por fila.")
filas_A = []
for i in range(len(b)):
    entrada_fila = input(f"Fila {i+1} separados por espacio: ")
    filas_A.append([float(x) for x in entrada_fila.split()])

A = np.array(filas_A)

#EJECUCION
try:
    num_vars = len(c)
    num_restr = len(b)

    # Paso 1: estandarizacion
    tablero_inicial = init_tablero(c, A, b)

    # Paso 2: solucion matematica
    tablero_final = simplex(tablero_inicial.copy(), num_vars, num_restr)

    mostrar_resultados(tablero_final, num_vars, num_restr)

except Exception as e:
    print(f"Error durante la ejecucon: {e}")

--- INGRESO DE DATOS ---
Ingrese los coeficientes de Z separados por espacio (Ej: 3 5): 3 5
Ingrese los valores del lado derecho (RHS) separados por espacio (Ej: 4 12 18): 4 12 18
Ingrese los coeficientes de las restricciones fila por fila.
Fila 1 separados por espacio: 1 2
Fila 2 separados por espacio: 0 2
Fila 3 separados por espacio: 3 2

Solucion Optima Encontrada en 2 iteraciones

Tablero Final:


,X1,X2,S1,S2,S3,RHS
R1,1.0,2.0,1.0,0.0,0.0,4.0
R2,0.0,2.0,0.0,1.0,0.0,12.0
R3,0.0,-4.0,-3.0,0.0,1.0,6.0
Z,0.0,1.0,3.0,0.0,0.0,12.0



Solucion:
X1 = 4.0000
X2 = 0.0000
Z Maximo = 12.0000


#**VERSIÓN #2**

In [ ]:
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

class InterfazPL:
    def __init__(self):
        # --- 1. CONFIGURACIÓN INICIAL ---
        self.lbl_titulo = widgets.HTML("<h3>Calculadora de Programación Lineal (Método Simplex)</h3>")

        self.tipo_opt = widgets.Dropdown(options=['Maximizar', 'Minimizar'], description='Objetivo:')
        self.num_vars = widgets.BoundedIntText(value=2, min=2, max=10, description='Variables:', layout=widgets.Layout(width='150px'))
        self.num_rest = widgets.BoundedIntText(value=3, min=1, max=10, description='Restricciones:', layout=widgets.Layout(width='150px'))

        self.btn_generar = widgets.Button(description='Generar Matriz', button_style='info')
        self.btn_generar.on_click(self.generar_campos)

        self.config_box = widgets.HBox([self.tipo_opt, self.num_vars, self.num_rest, self.btn_generar])

        # Áreas dinámicas
        self.matriz_area = widgets.VBox()
        self.output_area = widgets.Output()

        # Mostrar interfaz inicial
        display(self.lbl_titulo, self.config_box, self.matriz_area, self.output_area)

    def generar_campos(self, b):
        self.matriz_area.children = [] # Limpiar área anterior
        n_vars = self.num_vars.value
        n_rest = self.num_rest.value

        # --- 2. FUNCIÓN OBJETIVO (Z) ---
        lbl_z = widgets.HTML("<b>Función Objetivo (Coeficientes de Z):</b>")
        self.inputs_c = [widgets.FloatText(value=0, layout=widgets.Layout(width='60px')) for _ in range(n_vars)]

        elementos_z = []
        for i in range(n_vars):
            elementos_z.append(widgets.Label(f"X{i+1}:"))
            elementos_z.append(self.inputs_c[i])
        fila_z = widgets.HBox(elementos_z)

        # --- 3. RESTRICCIONES (A y b) ---
        lbl_restr = widgets.HTML("<br><b>Restricciones (Sujeto a):</b>")
        self.inputs_A = []
        self.inputs_signo = []
        self.inputs_b = []

        filas_ui = []
        for i in range(n_rest):
            fila_A = [widgets.FloatText(value=0, layout=widgets.Layout(width='60px')) for _ in range(n_vars)]
            signo = widgets.Dropdown(options=['<=', '>=', '='], value='<=', layout=widgets.Layout(width='60px'))
            rhs = widgets.FloatText(value=0, layout=widgets.Layout(width='60px'))

            self.inputs_A.append(fila_A)
            self.inputs_signo.append(signo)
            self.inputs_b.append(rhs)

            elementos_fila = [widgets.Label(f"R{i+1}: ")] + fila_A + [signo, rhs]
            filas_ui.append(widgets.HBox(elementos_fila))

        # --- 4. BOTÓN RESOLVER ---
        self.btn_resolver = widgets.Button(description='Resolver Problema', button_style='success', icon='check')
        self.btn_resolver.on_click(self.resolver_problema)

        self.matriz_area.children = [lbl_z, fila_z, lbl_restr] + filas_ui + [widgets.HTML("<br>"), self.btn_resolver]

    def resolver_problema(self, b):
        with self.output_area:
            clear_output(wait=True)
            print("Procesando...")

            es_max = (self.tipo_opt.value == 'Maximizar')
            vector_c = np.array([inp.value for inp in self.inputs_c])
            matriz_A = np.array([[inp.value for inp in fila] for fila in self.inputs_A])
            vector_signos = np.array([inp.value for inp in self.inputs_signo])
            vector_b = np.array([inp.value for inp in self.inputs_b])

            try:
                # 1. Estandarización F.O.
                c_calc = vector_c.copy()
                if not es_max:
                    c_calc = -c_calc

                # 2. Inicializar Tablero
                tablero_inicial, nombres_columnas, variables_base = self.init_tablero_completo(
                    c_calc, matriz_A, vector_b, vector_signos
                )

                # 3. Simplex Paso a Paso
                tablero_final = self.simplex_paso_paso(tablero_inicial, nombres_columnas, variables_base)

                # 4. Resultados Finales
                print("\n" + "="*40)
                print("SOLUCIÓN ÓPTIMA FINAL")
                print("="*40)
                self.mostrar_resultados_finales(tablero_final, variables_base)

                # 5. Método Gráfico (Pendiente)
                if len(vector_c) == 2:
                    print("\n" + "="*40)
                    print("MÉTODO GRÁFICO (2 Variables detectadas)")
                    print("="*40)

            except Exception as e:
                print(f"Error al resolver el problema: {e}")

    # ===============================================
    # FUNCIONES MATEMÁTICAS COMO MÉTODOS DE LA CLASE
    # ===============================================

    def init_tablero_completo(self, c, A, b, signos):
        num_restr = len(b)
        num_vars = len(c)

        # Para esta versión, asumimos solo '<=' (Holguras).
        # Nota: agregar luego la lógica de la Gran M para '>=' y '='
        num_holguras = np.sum(signos == '<=')
        total_vars_adicionales = num_holguras

        filas = num_restr + 1
        columnas = num_vars + total_vars_adicionales + 1
        tablero = np.zeros((filas, columnas))

        # Insertar matriz A y vector b
        tablero[:num_restr, :num_vars] = A
        tablero[:num_restr, -1] = b
        tablero[-1, :num_vars] = -c

        nombres_columnas = [f"X{i+1}" for i in range(num_vars)]
        variables_base = []

        col_actual = num_vars
        for i, signo in enumerate(signos):
            if signo == '<=':
                tablero[i, col_actual] = 1
                nombres_columnas.append(f"S{i+1}")
                variables_base.append(f"S{i+1}")
                col_actual += 1

        nombres_columnas.append("RHS")
        return tablero, nombres_columnas, variables_base

    def simplex_paso_paso(self, tablero, columnas, vars_base):
        iteracion = 0
        while True:
            # Imprimir tablero paso a paso
            df = pd.DataFrame(np.round(tablero, 4), columns=columnas, index=vars_base + ["Z"])
            print(f"\n--- TABLERO ITERACIÓN {iteracion} ---")
            display(df)

            # Condición de parada
            if np.all(tablero[-1, :-1] >= -1e-8):
                break

            columna_pivote = np.argmin(tablero[-1, :-1])
            rhs = tablero[:-1, -1]
            columna_entrada = tablero[:-1, columna_pivote]

            cocientes = np.divide(rhs, columna_entrada, out=np.full_like(rhs, np.inf), where=columna_entrada > 1e-8)

            if np.all(np.isinf(cocientes)):
                raise ValueError("El problema no está acotado. No tiene solución finita.")

            fila_pivote = np.argmin(cocientes)

            print(f"➜ Entra: {columnas[columna_pivote]} | Sale: {vars_base[fila_pivote]} | Pivote: {tablero[fila_pivote, columna_pivote]:.4f}")

            vars_base[fila_pivote] = columnas[columna_pivote]

            elemento_pivote = tablero[fila_pivote, columna_pivote]
            tablero[fila_pivote, :] = tablero[fila_pivote, :] / elemento_pivote

            for f in range(tablero.shape[0]):
                if f != fila_pivote:
                    factor = tablero[f, columna_pivote]
                    tablero[f, :] = tablero[f, :] - factor * tablero[fila_pivote, :]

            iteracion += 1

        return tablero

    def mostrar_resultados_finales(self, tablero, vars_base):
        print("\nVariables Básicas:")
        for i, var in enumerate(vars_base):
            print(f"{var} = {tablero[i, -1]:.4f}")

        print(f"\nZ Máximo = {tablero[-1, -1]:.4f}")

# Ejecutar la aplicación
app = InterfazPL()

HTML(value='<h3>Calculadora de Programación Lineal (Método Simplex)</h3>')

VBox()

Output()

# **VERSIÓN #3**

In [ ]:
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull

class InterfazPL:
    def __init__(self):
        # --- 1. CONFIGURACIÓN INICIAL ---
        self.lbl_titulo = widgets.HTML("<h3>Calculadora de Programación Lineal (Multivariable)</h3>")

        self.tipo_opt = widgets.Dropdown(options=['Maximizar', 'Minimizar'], description='Objetivo:')
        # Se habilita el control para permitir más de 2 variables
        self.num_vars = widgets.BoundedIntText(value=2, min=2, max=10, description='Variables:', layout=widgets.Layout(width='150px'))
        self.num_rest = widgets.BoundedIntText(value=3, min=1, max=10, description='Restricciones:', layout=widgets.Layout(width='150px'))

        self.btn_generar = widgets.Button(description='Generar Matriz', button_style='info')
        self.btn_generar.on_click(self.generar_campos)

        # Etiqueta informativa sobre la condicionalidad del gráfico
        self.lbl_vars_info = widgets.HTML("<small><i>(Gráfico solo disponible para 2 variables)</i></small>", layout=widgets.Layout(margin='0 10px 0 5px'))

        # Orden correcto de instanciación para evitar el AttributeError
        self.config_box = widgets.HBox([self.tipo_opt, self.num_vars, self.lbl_vars_info, self.num_rest, self.btn_generar])

        # Áreas dinámicas de visualización
        self.matriz_area = widgets.VBox()
        self.output_area = widgets.Output()

        # Mostrar interfaz inicial
        display(self.lbl_titulo, self.config_box, self.matriz_area, self.output_area)

    def generar_campos(self, b):
        self.matriz_area.children = [] # Limpiar el área de entrada anterior
        n_vars = self.num_vars.value   # Captura dinámica de variables (2 o más)
        n_rest = self.num_rest.value   # Captura dinámica de restricciones

        # --- 2. FUNCIÓN OBJETIVO (Z) ---
        lbl_z = widgets.HTML("<b>Función Objetivo (Coeficientes de Z):</b> Z = c1*X1 + c2*X2 + ... + cn*Xn")
        self.inputs_c = [widgets.FloatText(value=0, layout=widgets.Layout(width='60px')) for _ in range(n_vars)]

        elementos_z = []
        for i in range(n_vars):
            elementos_z.append(widgets.Label(f"c{i+1} (X{i+1}):"))
            elementos_z.append(self.inputs_c[i])
        fila_z = widgets.HBox(elementos_z)

        # --- 3. RESTRICCIONES (A y b) ---
        lbl_restr = widgets.HTML("<br><b>Restricciones estructurales:</b><br><small>Asumimos condiciones de no negatividad (Xi >= 0)</small>")
        self.inputs_A = []
        self.inputs_signo = []
        self.inputs_b = []

        filas_ui = []
        for i in range(n_rest):
            fila_A = [widgets.FloatText(value=0, layout=widgets.Layout(width='60px')) for _ in range(n_vars)]
            signo = widgets.Dropdown(options=['<=', '>=', '='], value='<=', layout=widgets.Layout(width='60px'))
            rhs = widgets.FloatText(value=0, layout=widgets.Layout(width='60px'))

            self.inputs_A.append(fila_A)
            self.inputs_signo.append(signo)
            self.inputs_b.append(rhs)

            elementos_fila = [widgets.Label(f"R{i+1}: ")] + fila_A + [signo, rhs]
            filas_ui.append(widgets.HBox(elementos_fila))

        # --- 4. BOTÓN RESOLVER ---
        self.btn_resolver = widgets.Button(description='Resolver Problema', button_style='success', icon='check')
        self.btn_resolver.on_click(self.resolver_problema)

        self.matriz_area.children = [lbl_z, fila_z, lbl_restr] + filas_ui + [widgets.HTML("<br>"), self.btn_resolver]

    def resolver_problema(self, b):
        with self.output_area:
            clear_output(wait=True)
            print("⏳ Extrayendo matrices y procesando optimización...")

            es_max = (self.tipo_opt.value == 'Maximizar')
            vector_c = np.array([inp.value for inp in self.inputs_c])
            matriz_A = np.array([[inp.value for inp in fila] for fila in self.inputs_A])
            vector_signos = np.array([inp.value for inp in self.inputs_signo])
            vector_b = np.array([inp.value for inp in self.inputs_b])

            try:
                # --- 1. RESOLUCIÓN GENERAL POR MÉTODO SIMPLEX ---
                print("\n" + "="*50)
                print("📗 ALGORITMO SIMPLEX: EVOLUCIÓN TABULAR PASO A PASO")
                print("="*50)

                c_calc = vector_c.copy()
                if not es_max:
                    c_calc = -c_calc # Estandarización algebraica de la F.O.

                tablero_inicial, nombres_columnas, variables_base = self.init_tablero_completo(c_calc, matriz_A, vector_b, vector_signos)
                tablero_final = self.simplex_paso_paso(tablero_inicial, nombres_columnas, variables_base)
                self.mostrar_resultados_finales_simplex(tablero_final, nombres_columnas, len(vector_c))

                # --- 2. EVALUACIÓN CONDICIONAL DEL MÉTODO GRÁFICO ---
                if len(vector_c) == 2:
                    print("\n" + "="*50)
                    print("📊 ANÁLISIS GEOMÉTRICO: MÉTODO GRÁFICO (2D)")
                    print("="*50)
                    self.ejecutar_metodo_grafico(vector_c, matriz_A, vector_b, vector_signos, es_max)
                else:
                    print("\n" + "="*50)
                    print("ℹ️ MÉTODO GRÁFICO NO DISPONIBLE")
                    print("="*50)
                    print(f"La representación geométrica bidimensional está restringida a problemas de 2 variables.")
                    print(f"Tu modelo actual cuenta con {len(vector_c)} variables de decisión, por lo que se resolvió exclusivamente vía matrices multidimensionales (Simplex).")

            except Exception as e:
                print(f"❌ Fallo crítico durante la ejecución del algoritmo: {e}")

    # =========================================================================
    # --- PROCESAMIENTO MATRICIAL SIMPLEX ---
    # =========================================================================
    def init_tablero_completo(self, c, A, b, signos):
        num_restr, num_vars = A.shape
        total_vars = num_vars + num_restr
        tablero = np.zeros((num_restr + 1, total_vars + 1))

        tablero[:num_restr, :num_vars] = A
        tablero[:num_restr, num_vars:total_vars] = np.eye(num_restr) # Holguras estándar (<=)
        tablero[:num_restr, -1] = b
        tablero[-1, :num_vars] = -c

        columnas = [f"X{i+1}" for i in range(num_vars)] + [f"S{i+1}" for i in range(num_restr)] + ["RHS"]
        base = [f"S{i+1}" for i in range(num_restr)]
        return tablero, columnas, base

    def simplex_paso_paso(self, tablero, columnas, vars_base):
        iteracion = 0
        while True:
            # Renderizado tabular usando DataFrames de Pandas
            df = pd.DataFrame(np.round(tablero, 3), columns=columnas, index=vars_base + ["Z"])
            print(f"\n--- TABLERO ITERACIÓN {iteracion} ---")
            display(df)

            if np.all(tablero[-1, :-1] >= -1e-8):
                break # Condición de optimalidad alcanzada

            piv_c = np.argmin(tablero[-1, :-1])
            rhs = tablero[:-1, -1]
            piv_col_val = tablero[:-1, piv_c]

            ratios = np.divide(rhs, piv_col_val, out=np.full_like(rhs, np.inf), where=piv_col_val > 1e-8)
            if np.all(np.isinf(ratios)):
                raise ValueError("El espacio de soluciones no está acotado (Z óptima tiende al infinito).")

            piv_f = np.argmin(ratios)
            print(f"➜ Transición: Entra la variable {columnas[piv_c]} a la base sustituyendo a {vars_base[piv_f]}. Valor Pivote: {tablero[piv_f, piv_c]:.2f}")

            vars_base[piv_f] = columnas[piv_c]
            tablero[piv_f, :] /= tablero[piv_f, piv_c]

            for f in range(tablero.shape[0]):
                if f != piv_f:
                    tablero[f, :] -= tablero[f, piv_c] * tablero[piv_f, :]
            iteracion += 1
        return tablero

    def mostrar_resultados_finales_simplex(self, tablero, columnas, n_vars):
        print("\n🏆 VALORES ÓPTIMOS DE LAS VARIABLES ENCONTRADOS:")
        # Escaneo de columnas para extraer los valores de las variables estructurales X
        for idx_var in range(n_vars):
            columna = tablero[:, idx_var]
            # Validamos si es una variable básica matemática (un 1 y el resto ceros)
            if np.sum(columna == 1) == 1 and np.sum(columna == 0) == len(columna) - 1:
                fila_uno = np.where(columna == 1)[0][0]
                valor = tablero[fila_uno, -1]
            else:
                valor = 0.0 # Las variables no básicas valen cero
            print(f"📊 Variable X{idx_var+1} = {valor:.2f}")

        print(f"💰 Valor de la Función Objetivo Óptima (Z) = {tablero[-1, -1]:.2f}")

    # =========================================================================
    # --- ENGINE GEOMÉTRICO (MÉTODO GRÁFICO 2D) ---
    # =========================================================================
    def ejecutar_metodo_grafico(self, c, A, b, signos, es_max):
        try:
            vertices_factibles = self.hallar_vertices_region_factible(A, b, signos)
            if vertices_factibles.size == 0:
                print("❌ Conflicto geométrico: La intersección de restricciones genera una Región Factible Vacía.")
                return

            # Ordenación perimetral de coordenadas para el sombreado de polígonos convexos
            if vertices_factibles.shape[0] > 2:
                hull = ConvexHull(vertices_factibles)
                vertices_ordenados = vertices_factibles[hull.vertices]
            else:
                vertices_ordenados = vertices_factibles

            valores_z = vertices_ordenados @ c
            piv_opt = np.argmax(valores_z) if es_max else np.argmin(valores_z)

            punto_optimo = vertices_ordenados[piv_opt]
            z_optimo = valores_z[piv_opt]

            self.graficar_solucion(vertices_ordenados, punto_optimo, z_optimo, c, A, b, signos, es_max)

        except Exception as e:
            print(f"⚠️ Error interno en el renderizado de gráficos: {e}")

    def hallar_vertices_region_factible(self, A, b, signos):
        n_restr = A.shape[0]
        lineas_A = np.vstack([A, [1, 0], [0, 1]]) # Fronteras de restricciones más los ejes coordenados
        lineas_b = np.hstack([b, 0, 0])
        puntos = []

        for i in range(len(lineas_b)):
            for j in range(i + 1, len(lineas_b)):
                M = np.vstack([lineas_A[i], lineas_A[j]])
                rhs = np.array([lineas_b[i], lineas_b[j]])
                try:
                    p = np.linalg.solve(M, rhs)
                    Tol = 1e-6
                    factible = True
                    for k in range(n_restr):
                        calc = A[k] @ p
                        if signos[k] == '<=' and calc > b[k] + Tol: factible = False; break
                        if signos[k] == '>=' and calc < b[k] - Tol: factible = False; break
                        if signos[k] == '=' and not np.isclose(calc, b[k], atol=Tol): factible = False; break

                    if p[0] < -Tol or p[1] < -Tol:
                        factible = False
                    if factible:
                        puntos.append(p)
                except np.linalg.LinAlgError:
                    pass

        return np.unique(np.array(puntos), axis=0) if puntos else np.array([])

    def graficar_solucion(self, vertices, p_opt, z_opt, c, A, b, signos, es_max):
        plt.figure(figsize=(9, 8))
        max_v = np.max(vertices) * 1.3 if vertices.size > 0 else 10
        xlim = (0, max(max_v, 10)); ylim = (0, max(max_v, 10))
        x_vals = np.linspace(xlim[0], xlim[1], 400)

        # Trazado de rectas frontera correspondientes a las inecuaciones
        cols = ['blue', 'green', 'purple', 'cyan', 'darkgreen']
        for i in range(A.shape[0]):
            a1, a2 = A[i]
            color = cols[i % len(cols)]
            if a2 != 0:
                plt.plot(x_vals, (b[i] - a1 * x_vals) / a2, label=f"Restricción {i+1}", color=color, linestyle='--', alpha=0.8)
            else:
                plt.axvline(x=b[i]/a1, label=f"Restricción {i+1}", color=color, linestyle='--', alpha=0.8)

        # Sombreado de la Región de Soluciones Factibles
        plt.fill(vertices[:, 0], vertices[:, 1], color='gold', alpha=0.3, label='Región Factible')

        # Trazado de las Rectas de Isoutilidad / Isocosto
        c1, c2 = c
        if c2 != 0:
            z_base = z_opt * 0.2 if z_opt != 0 else 2
            plt.plot(x_vals, (z_base - c1 * x_vals) / c2, color='darkgray', linestyle=':', alpha=0.6, label='F.O. Dinámica (Desplazamiento)')
            plt.plot(x_vals, (z_opt - c1 * x_vals) / c2, color='crimson', linestyle='-', linewidth=2, label='Recta de Isoutilidad ÓPTIMA')
        else:
            plt.axvline(x=z_opt/c1, color='crimson', linestyle='-', linewidth=2, label='Recta de Isoutilidad ÓPTIMA')

        # Resaltado del Punto de Abandono (Vértice Óptimo Extremo)
        print(f"➜ 📍 Coordenada límite de contacto (Vértice Óptimo): ({p_opt[0]:.2f}, {p_opt[1]:.2f})")
        plt.scatter(p_opt[0], p_opt[1], color='crimson', s=180, edgecolors='black', zorder=5)
        plt.annotate(f" ÓPTIMO (Punto de abandono)\n X1={p_opt[0]:.2f}, X2={p_opt[1]:.2f}\n Z={z_opt:.2f}",
                     p_opt, xytext=(p_opt[0]+0.3, p_opt[1]+0.3), fontsize=10, fontweight='bold', bbox=dict(boxstyle="round,pad=0.3", fc="yellow", alpha=0.5))

        plt.xlim(xlim); plt.ylim(ylim)
        plt.xlabel('Eje X1 (Variable de decisión 1)'); plt.ylabel('Eje X2 (Variable de decisión 2)')
        plt.title(f"Análisis Espacial del Modelo Lineal ({self.tipo_opt.value})")
        plt.axhline(0, color='black', linewidth=1.2); plt.axvline(0, color='black', linewidth=1.2)
        plt.grid(True, linestyle=':', alpha=0.5)
        plt.legend(loc='upper right')
        display(plt.gcf())
        plt.close()

# Inicialización y renderizado automático del programa
app = InterfazPL()

HTML(value='<h3>Calculadora de Programación Lineal (Multivariable)</h3>')

VBox()

Output()

# **HARMONY SEARCH**

El algoritmo de **Búsqueda Armónica** (Harmony Search, HS) es una metaheurística de optimización propuesta por Geem et al. (2001), inspirada en el proceso de improvisación musical. Un músico busca una armonía perfecta de la misma manera que un algoritmo busca la solución óptima de un problema.

**Componentes principales:**
- **HMS** (*Harmony Memory Size*): número de soluciones que se guardan en memoria.
- **HMCR** (*Harmony Memory Consideration Rate*): probabilidad de tomar un valor de la memoria en lugar de uno aleatorio.
- **PAR** (*Pitch Adjustment Rate*): probabilidad de ajustar ligeramente el valor seleccionado de la memoria.
- **BW** (*Bandwidth*): magnitud del ajuste de tono como fracción del rango de la variable.
- **NI** (*Number of Improvisations*): cantidad total de iteraciones.

**Flujo del algoritmo:**
1. Inicializar la Memoria Armónica (HM) con `HMS` soluciones aleatorias dentro de los límites.
2. Improvisar una nueva armonía: para cada variable, con probabilidad HMCR se selecciona un valor de la HM (y con probabilidad PAR se ajusta con ±BW), de lo contrario se genera aleatoriamente.
3. Si la nueva armonía es mejor que la peor en la HM, la reemplaza.
4. Repetir hasta completar NI improvisaciones.

Para problemas con restricciones se aplica el **método de penalización**: las soluciones que violan alguna restricción reciben una penalidad proporcional a la magnitud de la violación.

In [ ]:
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt


class HarmonySearch:
    def __init__(self):
        self.titulo = widgets.HTML("<h3>Optimizacion por Busqueda Armonica (Harmony Search)</h3>")

        self.tipo_opt = widgets.Dropdown(
            options=['Maximizar', 'Minimizar'],
            description='Objetivo:',
            layout=widgets.Layout(width='160px')
        )
        self.num_vars = widgets.BoundedIntText(
            value=2, min=1, max=10,
            description='Variables:',
            layout=widgets.Layout(width='140px')
        )
        self.num_rest = widgets.BoundedIntText(
            value=3, min=1, max=15,
            description='Restricciones:',
            layout=widgets.Layout(width='165px')
        )
        self.btn_generar = widgets.Button(description='Generar Campos', button_style='info')
        self.btn_generar.on_click(self.generar_campos)

        self.hms_w  = widgets.BoundedIntText(value=20,   min=5,    max=100,    description='HMS:',         layout=widgets.Layout(width='130px'))
        self.hmcr_w = widgets.BoundedFloatText(value=0.85, min=0.5, max=0.99, step=0.01, description='HMCR:', layout=widgets.Layout(width='130px'))
        self.par_w  = widgets.BoundedFloatText(value=0.35, min=0.01, max=0.99, step=0.01, description='PAR:',  layout=widgets.Layout(width='130px'))
        self.bw_w   = widgets.BoundedFloatText(value=0.05, min=0.001, max=1.0, step=0.005, description='BW:',  layout=widgets.Layout(width='130px'))
        self.ni_w   = widgets.BoundedIntText(value=5000, min=100,  max=100000, description='Iteraciones:', layout=widgets.Layout(width='165px'))

        lbl_params = widgets.HTML("<b>Parametros Harmony Search:</b>")
        params_box = widgets.HBox([self.hms_w, self.hmcr_w, self.par_w, self.bw_w, self.ni_w])
        config_box = widgets.HBox([self.tipo_opt, self.num_vars, self.num_rest, self.btn_generar])

        self.area_entrada = widgets.VBox()
        self.salida = widgets.Output()

        display(self.titulo, config_box, lbl_params, params_box, self.area_entrada, self.salida)

    def generar_campos(self, b):
        self.area_entrada.children = []
        n = self.num_vars.value
        m = self.num_rest.value

        lbl_fo = widgets.HTML("<b>Funcion Objetivo (Coeficientes de Z):</b>")
        self.c_inputs = [widgets.FloatText(value=0.0, layout=widgets.Layout(width='65px')) for _ in range(n)]
        fila_c = []
        for i in range(n):
            fila_c += [widgets.Label(f"X{i+1}:"), self.c_inputs[i]]

        lbl_r = widgets.HTML("<b>Restricciones (Sujeto a):</b>")
        self.A_inputs  = []
        self.sig_inputs = []
        self.b_inputs  = []
        filas_r = []

        for i in range(m):
            fila = [widgets.FloatText(value=0.0, layout=widgets.Layout(width='65px')) for _ in range(n)]
            sig  = widgets.Dropdown(options=['<=', '>=', '='], value='<=', layout=widgets.Layout(width='65px'))
            rhs  = widgets.FloatText(value=0.0, layout=widgets.Layout(width='65px'))
            self.A_inputs.append(fila)
            self.sig_inputs.append(sig)
            self.b_inputs.append(rhs)
            elem = [widgets.Label(f"R{i+1}: ")] + fila + [sig, rhs]
            filas_r.append(widgets.HBox(elem))

        lbl_ub = widgets.HTML("<b>Limite superior de busqueda por variable (lb=0 asumido):</b>")
        self.ub_inputs = [widgets.FloatText(value=20.0, layout=widgets.Layout(width='65px')) for _ in range(n)]
        fila_ub = []
        for i in range(n):
            fila_ub += [widgets.Label(f"ub X{i+1}:"), self.ub_inputs[i]]

        btn = widgets.Button(description='Ejecutar Harmony Search', button_style='success', icon='check')
        btn.on_click(self.ejecutar)

        self.area_entrada.children = (
            [lbl_fo, widgets.HBox(fila_c), lbl_r]
            + filas_r
            + [lbl_ub, widgets.HBox(fila_ub), widgets.HTML("<br>"), btn]
        )

    def _penalidad(self, x, A, b, signos):
        total = 0.0
        for i in range(len(b)):
            v = float(A[i] @ x)
            if signos[i] == '<=' and v > b[i]:
                total += v - b[i]
            elif signos[i] == '>=' and v < b[i]:
                total += b[i] - v
            elif signos[i] == '=':
                total += abs(v - b[i])
        return total

    def _fitness(self, x, c, A, b, signos, es_max, M=1e6):
        z   = float(c @ x)
        pen = self._penalidad(x, A, b, signos)
        return (z - pen * M) if es_max else -(z + pen * M)

    def ejecutar(self, b):
        with self.salida:
            clear_output(wait=True)

            es_max  = self.tipo_opt.value == 'Maximizar'
            c       = np.array([w.value for w in self.c_inputs])
            A       = np.array([[w.value for w in fila] for fila in self.A_inputs])
            signos  = [w.value for w in self.sig_inputs]
            b_vec   = np.array([w.value for w in self.b_inputs])
            ub      = np.array([w.value for w in self.ub_inputs])
            lb      = np.zeros(len(c))

            HMS  = self.hms_w.value
            HMCR = self.hmcr_w.value
            PAR  = self.par_w.value
            BW   = self.bw_w.value
            NI   = self.ni_w.value
            n    = len(c)
            M    = 1e6

            print("Ejecutando Harmony Search...")
            print(f"HMS={HMS}  HMCR={HMCR}  PAR={PAR}  BW={BW}  NI={NI}")
            print("-" * 55)

            np.random.seed(42)
            HM  = np.random.uniform(lb, ub, (HMS, n))
            fit = np.array([self._fitness(HM[k], c, A, b_vec, signos, es_max, M) for k in range(HMS)])

            mejor_fit  = float(np.max(fit))
            mejor_sol  = HM[int(np.argmax(fit))].copy()
            hist_fit   = [mejor_fit]

            mejor_factible   = None
            mejor_z_factible = -np.inf if es_max else np.inf

            for k in range(HMS):
                if self._penalidad(HM[k], A, b_vec, signos) < 1e-6:
                    z_k = float(c @ HM[k])
                    if (es_max and z_k > mejor_z_factible) or (not es_max and z_k < mejor_z_factible):
                        mejor_z_factible = z_k
                        mejor_factible   = HM[k].copy()

            for it in range(NI):
                nueva = np.empty(n)
                for j in range(n):
                    if np.random.rand() < HMCR:
                        nueva[j] = HM[np.random.randint(HMS), j]
                        if np.random.rand() < PAR:
                            nueva[j] += np.random.uniform(-BW, BW) * (ub[j] - lb[j])
                    else:
                        nueva[j] = np.random.uniform(lb[j], ub[j])
                    nueva[j] = float(np.clip(nueva[j], lb[j], ub[j]))

                fit_nueva = self._fitness(nueva, c, A, b_vec, signos, es_max, M)
                idx_peor  = int(np.argmin(fit))

                if fit_nueva > fit[idx_peor]:
                    HM[idx_peor]  = nueva.copy()
                    fit[idx_peor] = fit_nueva
                    if fit_nueva > mejor_fit:
                        mejor_fit = fit_nueva
                        mejor_sol = nueva.copy()

                if self._penalidad(nueva, A, b_vec, signos) < 1e-6:
                    z_n = float(c @ nueva)
                    if (es_max and z_n > mejor_z_factible) or (not es_max and z_n < mejor_z_factible):
                        mejor_z_factible = z_n
                        mejor_factible   = nueva.copy()

                hist_fit.append(mejor_fit)

            sol_final = mejor_factible if mejor_factible is not None else mejor_sol
            pen_final = self._penalidad(sol_final, A, b_vec, signos)
            z_final   = float(c @ sol_final)

            print("\nRESULTADOS FINALES")
            print("=" * 55)

            df_res = pd.DataFrame({
                'Variable':     [f'X{i+1}' for i in range(n)],
                'Valor Optimo': np.round(sol_final, 4)
            })
            display(df_res)

            tipo_z = 'Maximo' if es_max else 'Minimo'
            print(f"\nZ {tipo_z} = {z_final:.4f}")

            if pen_final < 1e-4:
                print("Solucion factible: todas las restricciones se cumplen.")
            else:
                print(f"Advertencia: violacion acumulada = {pen_final:.6f}")
                print("Aumente NI o HMS para mejorar la convergencia.")

            print("\nVerificacion de restricciones:")
            for i in range(len(b_vec)):
                lhs = float(A[i] @ sol_final)
                ok  = (
                    (signos[i] == '<=' and lhs <= b_vec[i] + 1e-4) or
                    (signos[i] == '>=' and lhs >= b_vec[i] - 1e-4) or
                    (signos[i] == '=' and abs(lhs - b_vec[i]) <= 1e-4)
                )
                print(f"  R{i+1}: {lhs:.4f} {signos[i]} {b_vec[i]}  ->  {'OK' if ok else 'VIOLADA'}")

            # --- GRAFICO DE CONVERGENCIA ---
            fig, axes = plt.subplots(1, 2, figsize=(13, 4))

            axes[0].plot(hist_fit, color='steelblue', linewidth=1.2)
            axes[0].set_xlabel('Improvisacion')
            axes[0].set_ylabel('Mejor Fitness')
            axes[0].set_title('Convergencia - Harmony Search')
            axes[0].grid(True, linestyle=':', alpha=0.5)

            # Distribucion final de la memoria armonica
            if n >= 2:
                axes[1].scatter(HM[:, 0], HM[:, 1], color='steelblue', alpha=0.6, label='Memoria armonica')
                axes[1].scatter(sol_final[0], sol_final[1], color='crimson', s=120, zorder=5, label=f'Optimo ({sol_final[0]:.2f}, {sol_final[1]:.2f})')
                axes[1].set_xlabel('X1')
                axes[1].set_ylabel('X2')
                axes[1].set_title('Distribucion Final de la Memoria Armonica')
                axes[1].legend()
                axes[1].grid(True, linestyle=':', alpha=0.5)
            else:
                vals = np.array([float(c @ HM[k]) for k in range(HMS)])
                axes[1].hist(vals, bins=10, color='steelblue', alpha=0.7, edgecolor='black')
                axes[1].set_xlabel('Z')
                axes[1].set_ylabel('Frecuencia')
                axes[1].set_title('Distribucion de Z en la Memoria Final')
                axes[1].grid(True, linestyle=':', alpha=0.5)

            plt.tight_layout()
            display(plt.gcf())
            plt.close()


app_hs = HarmonySearch()
